# B4 Experiment — Community-concentrated opponent

B4 selects the largest Louvain community on the undirected projection, then ranks by out-degree.

Target: 5 K values × 6 objectives × 2 modes = **60 configurations**. SSA is run once per (K, objective) and its saved seed set is shared by SSA-only and SSA+Swap. Results are checkpointed and written under `results/<run_id>/<scenario>/`.

In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
for p in [ROOT, ROOT / 'common', ROOT / 'cpp_engine']:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from network import NETWORK_CSV, NETWORK_METADATA_V2, load_network, verify_fingerprint
from experiment import ExperimentConfig, ExperimentRunner, MODE_SSA, MODE_SWAP, default_run_id
from objectives import VARIANTS

SCENARIO = 'B4'


## Shared-network fingerprint check

In [ ]:
check = verify_fingerprint(NETWORK_CSV, NETWORK_METADATA_V2)
print(json.dumps(check, indent=2))
assert check['ok'], 'Network fingerprint mismatch: stop before running experiments.'
df, G, nodes = load_network(NETWORK_CSV)
print(f'Shared network: {len(nodes)} nodes, {G.number_of_edges()} directed arcs')


## Configuration

`RUN_MODE` defaults to `'none'`, so **Run All does not launch an experiment**. Set it to `'smoke'` for a tiny pipeline check or `'full'` for the 60-configuration scenario. A full run additionally requires `CONFIRM_FULL_RUN = True`.

In [ ]:
RUN_MODE = 'none'          # 'none', 'smoke', or 'full'
CONFIRM_FULL_RUN = False   # must be True for RUN_MODE == 'full'
RUN_ID = None              # set a name to resume a specific run

K_VALUES = (10, 20, 30, 40, 50)
OBJECTIVE_IDS = tuple(VARIANTS.keys())
MODES = (MODE_SSA, MODE_SWAP)
SSA_POPULATION = 12
SSA_ITERATIONS = 10
SSA_MUTATION_RATE = 0.20
OPTIMIZATION_MC_RUNS = 15
FINAL_MC_RUNS = 100
SWAP_MAX_TESTS = 100
THREADS = 0                # 0 = OpenMP runtime default
RESUME = True

print('Planned configurations:', len(K_VALUES) * len(OBJECTIVE_IDS) * len(MODES))


## Safety confirmation

In [ ]:
if RUN_MODE == 'full' and not CONFIRM_FULL_RUN:
    raise RuntimeError('Full run not confirmed. Set CONFIRM_FULL_RUN=True after reviewing the plan.')
if RUN_MODE == 'none':
    print('Nothing will run. Set RUN_MODE to smoke or full.')


## Run (checkpointed and resumable)

In [ ]:
if RUN_MODE != 'none':
    run_kind = 'SMOKE_TEST' if RUN_MODE == 'smoke' else 'full'
    run_id = RUN_ID or default_run_id(run_kind)
    if RUN_MODE == 'smoke':
        config = ExperimentConfig(
            run_id=run_id, run_kind='SMOKE_TEST', scenarios=(SCENARIO,),
            k_values=(10,), objective_ids=('O2',), modes=MODES,
            ssa_population=4, ssa_iterations=2, optimization_mc_runs=5,
            final_mc_runs=10, swap_max_tests=10, n_threads=THREADS,
            network_path=NETWORK_CSV, results_root=ROOT / 'results' / 'smoke',
            resume=RESUME,
        )
    else:
        config = ExperimentConfig(
            run_id=run_id, run_kind='full', scenarios=(SCENARIO,),
            k_values=K_VALUES, objective_ids=OBJECTIVE_IDS, modes=MODES,
            ssa_population=SSA_POPULATION, ssa_iterations=SSA_ITERATIONS,
            ssa_mutation_rate=SSA_MUTATION_RATE,
            optimization_mc_runs=OPTIMIZATION_MC_RUNS, final_mc_runs=FINAL_MC_RUNS,
            swap_max_tests=SWAP_MAX_TESTS, n_threads=THREADS,
            network_path=NETWORK_CSV, results_root=ROOT / 'results',
            resume=RESUME,
        )
    print('Run directory:', config.run_dir)
    runner = ExperimentRunner(config)
    manifest = runner.run_all()
    print(json.dumps(manifest, indent=2))


## Results table

In [ ]:
if RUN_MODE != 'none':
    import pandas as pd
    results = pd.read_csv(config.scenario_dir(SCENARIO) / 'experiment_results.csv')
    columns = ['K', 'objective_id', 'mode', 'spread_A', 'spread_B', 'net_advantage', 'blocked_B', 'objective_validity']
    print(results[columns].to_string(index=False))


## Quick view

In [ ]:
if RUN_MODE != 'none':
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(9, 5))
    for oid in results['objective_id'].unique():
        for mode in results['mode'].unique():
            s = results[(results['objective_id'] == oid) & (results['mode'] == mode)].sort_values('K')
            if not s.empty:
                ax.plot(s['K'], s['net_advantage'], marker='o', label=f'{oid} {mode}')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xlabel('K')
    ax.set_ylabel('Net Advantage')
    ax.set_title(f'{SCENARIO} quick view')
    ax.legend(fontsize=7)
    plt.show()


## CLI alternative (recommended for overnight runs)

```bash
python common/run_experiment.py --scenarios B4 --run-id run_YYYYMMDD_HHMMSS
```

Re-running the same `--run-id` resumes without duplicating completed configurations.